Preprocessing the DATA

In [19]:
import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv("/content/data/Facets Assignment.csv")
df.head()

,Facets
0,Risktaking
1,Naivety
2,Acidity
3,Democratic Leadership:
4,Common-sense


In [26]:
# Remove trailing spaces
df["Facets"] = df["Facets"].str.strip()

# Remove rows with null values in Facets
df = df.dropna(subset=["Facets"])

# Remove duplicates
df = df.drop_duplicates()

# Remove trailing colons and other punctuation at the end
df["Facets"] = df["Facets"].str.replace(r':+$', '', regex=True)

# Remove numbers and special prefixes (like "596.", "926.", etc.)
df["Facets"] = df["Facets"].str.replace(r'^\d+\.\s*', '', regex=True)

df.to_csv("/content/data/Facets-Assignment-Cleaned.csv", index=False)

Generating data (new columns : definition, rubirc and examples)

In [1]:
pip install google-generativeai

In [4]:
import os
import pandas as pd
import google.generativeai as genai
import json
import time
import re
from tqdm import tqdm
from google.colab import userdata

In [8]:
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)

# The script now reads from the CSV created in step 1
INPUT_CSV = '/content/data/Facets-Assignment-Cleaned.csv'
OUTPUT_CSV = '/content/data/fully_automated_dataset.csv'

# Configure the generation model to use JSON mode
generation_config = {
  "temperature": 0.7,
  "response_mime_type": "application/json",
}
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=generation_config
)

# --- Main Script ---

# 1. Load the cleaned facet names directly from the CSV
try:
    df = pd.read_csv(INPUT_CSV)
    # The list of facets is simply the 'Facet' column
    facets = df['Facets'].unique().tolist()
    print(f"Loaded {len(facets)} unique facets from '{INPUT_CSV}'.")
except FileNotFoundError:
    print(f"Error: Input file '{INPUT_CSV}' not found. Please run the preparation script first.")
    exit()

# 2. Loop through each facet and generate all data
all_generated_data = []
print("Starting fully automated data generation with Gemini 2.5 flash...")

for facet_name in tqdm(facets, desc="Generating Facets"):
    prompt = f"""
    You are an expert in psychology and behavioral science. Your task is to generate a complete data package for a given behavioral facet.

    **Facet:** "{facet_name}"

    **Instructions:**
    1.  Write a clear, concise definition of the facet.
    2.  Create a 5-point scoring rubric (1=low, 5=high) with short keywords.
    3.  Generate one realistic, conversational example sentence for EACH of the 5 scores.
    4.  Return a single, valid JSON object with the following structure:
        {{
            "definition": "A clear definition of the facet.",
            "rubric": "1: keyword | 2: keyword | 3: keyword | 4: keyword | 5: keyword",
            "examples": {{
                "1": "An example sentence for score 1.",
                "2": "An example sentence for score 2.",
                "3": "An example sentence for score 3.",
                "4": "An example sentence for score 4.",
                "5": "An example sentence for score 5."
            }}
        }}
    """

    try:
        response = model.generate_content(prompt)
        # The response.text will be a JSON string, which we parse
        data_package = json.loads(response.text)
        data_package['facet_name'] = facet_name # Add the facet name for context
        all_generated_data.append(data_package)
        time.sleep(7) # Respect API rate limits

    except Exception as e:
        print(f"\nCould not process '{facet_name}'. Error: {e}")
        time.sleep(10)

# 3. Flatten the generated data into a training-ready format
print("\nFlattening the generated data into a training dataset...")
training_rows = []
for package in all_generated_data:
    context = f"Facet: {package['facet_name']}; Definition: {package['definition']}; Rubric: {package['rubric']}"
    for score, turn in package['examples'].items():
        training_rows.append({
            'context': context,
            'conversation_turn': turn,
            'score': int(score)
        })

training_df = pd.DataFrame(training_rows)

# 4. Save the final dataset
training_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Generation complete!")
print(f"Generated {len(training_df)} training examples for {len(all_generated_data)} facets.")
print(f"Dataset saved to '{OUTPUT_CSV}'.")
print("\nIMPORTANT: Please open the CSV and spot-check the data for quality and consistency before training.")

Loaded 399 unique facets from '/content/data/Facets-Assignment-Cleaned.csv'.
Starting fully automated data generation with Gemini 2.5 flash...


Generating Facets:  63%|██████▎   | 251/399 [58:08<35:21, 14.34s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1310.46ms



Could not process 'Abasement'. Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 33.055141916s.


Generating Facets:  63%|██████▎   | 252/399 [58:20<32:54, 13.43s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1209.66ms



Could not process 'Chronic pain presence'. Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 21.811852757s.


...


Could not process 'Decency'. Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 42.691071995s.


Generating Facets: 100%|█████████▉| 398/399 [1:26:10<00:11, 11.30s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1486.91ms



Could not process 'Processed-food frequency'. Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 250
Please retry in 31.320965293s.


Generating Facets: 100%|██████████| 399/399 [1:26:22<00:00, 12.99s/it]


Flattening the generated data into a training dataset...

✅ Generation complete!
Generated 1255 training examples for 251 facets.
Dataset saved to '/content/data/fully_automated_dataset.csv'.

IMPORTANT: Please open the CSV and spot-check the data for quality and consistency before training.
